# FedHealth — Notebook 01: Exploratory Data Analysis

Goals: understand the three CBC datasets, **document the data-quality decisions** that shape the whole project, and produce the evidence you can show at defense. Key conclusions reached here:
1. The **anemia CBC dataset** is our multi-disease workhorse (it already contains real *Leukemia* rows) — but it needs cleaning (impossible values).
2. `biased_leukemia_dataset.csv` has **no predictive signal** and no Pakistani data → **excluded** from modelling; kept only as a case study.
3. The **alpha-thalassemia** dataset is **not CBC-separable** (AUC ~0.5) → Model B's dataset choice is a documented open decision.

In [ ]:
# --- Setup ------------------------------------------------------------------
# Make the shared fedhealth_ml package importable in Colab. Pick ONE option:
#
#   (A) Clone your project repo (recommended):
# !git clone https://github.com/<you>/fedhealth.git
# import sys; sys.path.append('/content/fedhealth/ml')
#
#   (B) Upload the `ml/` folder via the Files panel, then:
# import sys; sys.path.append('/content/ml')
#
#   (C) Mount Google Drive where you keep the repo:
# from google.colab import drive; drive.mount('/content/drive')
# import sys; sys.path.append('/content/drive/MyDrive/fedhealth/ml')
import sys
sys.path.append('/content/fedhealth/ml')  # <-- adjust to match your choice above

# Install anything Colab is missing (Colab already has numpy/pandas/sklearn/tf).
!pip -q install xgboost>=2.0 scipy joblib 2>/dev/null

# Where the three CSVs live. Upload them or point DATA_DIR at your Drive folder.
DATA_DIR = '/content'          # <-- put the CSVs here (or change this path)
ANEMIA = f'{DATA_DIR}/anemia_diagnosed_cbc_data_v4.csv'
THAL   = f'{DATA_DIR}/thalassemia_alphanorm.csv'
LEUK   = f'{DATA_DIR}/biased_leukemia_dataset.csv'

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from fedhealth_ml import schema, preprocess as pp
print('fedhealth_ml loaded. Model A features:', len(schema.MODEL_A_FEATURES))


## 1. Load all three datasets

In [ ]:
a = pd.read_csv(ANEMIA)
t = pd.read_csv(THAL)
l = pd.read_csv(LEUK)
print('anemia', a.shape, '| thal', t.shape, '| leukemia', l.shape)

## 2. Model A dataset — class distribution
Note this file is multi-class and already includes **Leukemia** and **Healthy** — this is why we source leukemia here rather than from the biased file.

In [ ]:
counts = a['Diagnosis'].value_counts()
print(counts)
counts.plot(kind='barh'); plt.title('Model A class distribution'); plt.xlabel('rows'); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 3. Data-quality: physiologically impossible values
Several CBC columns contain impossible entries (negative HGB/MCV, WBC differentials in the thousands). We quantify them here; `preprocess.py` turns them into NaN and median-imputes (fit on train only).

In [ ]:
desc = a[schema.MODEL_A_FEATURES].describe().T[['min','50%','max']]
print(desc.round(1))
bad = {}
for c in schema.MODEL_A_FEATURES:
    lo, hi = (schema.PLAUSIBLE_RANGES.get(c) or schema.PLAUSIBLE_RANGES[c.upper()])
    bad[c] = int(((a[c] < lo) | (a[c] > hi)).sum())
print('\nout-of-range counts per feature:'); print(pd.Series(bad)[lambda s: s>0])

## 4. Case study: why `biased_leukemia_dataset.csv` is excluded
If a feature carries signal, its mean should differ between Positive and Negative cases. Here **every** feature — including bone-marrow-blast %, the strongest possible marker — is identical across the label. The label is independent of the features.

In [ ]:
g = l.groupby('Leukemia_Status')
for f in ['Bone_Marrow_Blasts','WBC_Count','Platelet_Count','Hemoglobin_Level','Age']:
    m = g[f].mean()
    print(f'{f:20} pos={m.get("Positive",float("nan")):.1f}  neg={m.get("Negative",float("nan")):.1f}')
print('\nCountries (note: no Pakistan):', list(l['Country'].value_counts().index[:6]))

**Conclusion:** no learnable signal + no locally-representative data. Excluded from training; retained here as a data-validation / fairness demonstration (ties to the National AI Policy 2025 point in the scope).

## 5. Thalassemia dataset — the CBC-separability check
We test whether alpha-carrier vs normal can be told apart from CBC indices. (Requires nothing but sklearn.)

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
y = (t['phenotype'] == 'alpha carrier').astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
def auc(cols):
    pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                         LogisticRegression(max_iter=1000))
    p = cross_val_predict(pipe, t[cols], y, cv=cv, method='predict_proba')[:,1]
    return round(roc_auc_score(y, p), 3)
cbc = schema.MODEL_B_FEATURES
print('CBC-only AUC          :', auc(cbc))
print('CBC + HbA2/HbF AUC    :', auc(cbc + ['hba2','hbf']))
print('electrophoresis-only  :', auc(['hba','hba2','hbf']))

**Conclusion:** ~0.5 AUC on CBC → alpha *silent carriers* are essentially invisible on a blood count (clinically expected). Model B needs either a **beta-thalassemia-trait** CBC dataset (which *is* CBC-detectable) or to be folded into Model A. Decision tracked in Notebook 04.